In [1]:
import pandas as pd
import numpy as np
from db_queries import get_covariate_estimates, get_location_metadata
import os

In [2]:
data_path_in = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/child_mortality/training_data/2026_10_07.01/data.parquet"
no_cutoff_path_in = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/child_mortality/training_data/2026_10_07.01/data_no_10_yr_cutoff.parquet"

data_path_out = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/child_mortality/training_data/2026_10_14.01/data.parquet"
no_cutoff_path_out = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/child_mortality/training_data/2026_10_14.01/data_no_10_yr_cutoff.parquet"
os.makedirs(os.path.dirname(data_path_out), mode=0o777, exist_ok=True)

In [6]:
df = pd.read_parquet(no_cutoff_path_out)
df.head()

,ihme_loc_id,geospatial_id,psu,strata,line_id,hh_id,lat,long,lbd_admin2_id,indv_id,...,age_6_m,age_12_m,age_24_m,age_36_m,age_48_m,age_60_m,age_group,lower_bound,upper_bound,consumption_median_RT
0,HTI,131,131,16,1,475,18.461470,-74.193090,71070,19720_131_475_1_2_2005_2_2006_4_14_1_23_NA,...,0,0,0,0,0,0,age_3_m,1,3,2.536232
1,IND,334909,334909,66,1,36,27.844276,81.516017,72354,157050_334909_36_1_2_2014_10_2014_2_4_1_NA_NA,...,0,0,0,0,0,0,age_3_m,1,3,4.384727
2,PER,144,144,7,1,62,-16.321743,-71.584303,77660,270404_144_62_1_2_2005_9_2008_4_43_1_NA_NA,...,0,0,0,0,0,0,age_3_m,1,3,5.985589
3,IND,41129,41129,41121,2,52,21.456395,82.820709,71912,467681_41129_52_2_1_2016_9_2019_2_41_1_NA_NA,...,0,0,0,0,0,0,age_3_m,1,3,4.858633
4,KEN,496,496,29,2,17,-1.175506,37.943442,74935,529017_496_17_2_1_2014_3_2022_3_96_1_43_NA,...,0,0,0,0,0,0,age_3_m,1,3,3.401140


In [4]:
# Investigate DHS data recent years
df[df["ihme_loc_id"] == "IND"]["birth_year"].max()

2021

In [ ]:
df_cutoff = pd.read_parquet(data_path_out)
df_cutoff[df_cutoff["ihme_loc_id"] == "IND"]["birth_year"].max()

2021

Need data for children > 1 month

In [6]:
df["age_group"].value_counts()

age_1_m     8022327
age_3_m     7738977
age_6_m     7611559
age_12_m    7454178
age_24_m    7175291
age_36_m    6721529
age_48_m    6303667
age_60_m    5902229
Name: age_group, dtype: int64

In [7]:
df = df[df["age_group"] != "age_1_m"]
df["age_group"].value_counts()

age_3_m     7738977
age_6_m     7611559
age_12_m    7454178
age_24_m    7175291
age_36_m    6721529
age_48_m    6303667
age_60_m    5902229
Name: age_group, dtype: int64

In [8]:
print(len(df))
print(df["int_birth_year_diff_months"].max())

48907430
476


In [16]:
df["age_group"].unique()

array(['age_1_m', 'age_3_m', 'age_6_m', 'age_12_m', 'age_24_m',
       'age_36_m', 'age_48_m', 'age_60_m'], dtype=object)

In [9]:
cols_to_check = [
    "child_mortality",
    "consumption_pd_cumul",
    "int_birth_year_diff_months",
    "birth_year",
    "days_over_30C_monthly_cumul",
    "total_precipitation_cumul",
    "lat",
    "long",
    "consumption_median_RT",
]

for c in cols_to_check:
    print(f"{c}: {df[c].isna().sum()} missing values")

child_mortality: 0 missing values
consumption_pd_cumul: 0 missing values
int_birth_year_diff_months: 0 missing values
birth_year: 0 missing values
days_over_30C_monthly_cumul: 0 missing values
total_precipitation_cumul: 0 missing values
lat: 0 missing values
long: 0 missing values
consumption_median_RT: 0 missing values


Cut off int_birth_year_diff_months to max 10

In [10]:
df.to_parquet(no_cutoff_path_out, index=False)

In [11]:
df = df[df["int_birth_year_diff_months"] <= 120]
print(f"Number of rows after filtering: {len(df):,}")

Number of rows after filtering: 22,263,556


In [12]:
df.to_parquet(data_path_out, index=False)